# Continuous Downhill — Steady-Speed Braking (Jupyter Version)

**Goal:** Find braking force and braking power distributions for **constant-speed downhill** (steady-state, $a = 0$).

> Mirrors `continuous downhill.py` with interactive inputs and plots.
> All values are **front total + rear total** (both wheels per axle). Braking balances the gravity component $m g \sin\theta$, not inertia.

---
### How to use
1. Edit the **Inputs** cell (slope, speed, distance, geometry, masses).
2. Run all cells.
3. Check the printed tables and charts.

Toggle `use_widgets = True` for sliders if `ipywidgets` is installed.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed — sliders disabled")

## 1 — Inputs (edit me)

* `constant_speed` — steady downhill speed along slope (m/s)
* `downhill_angle` — slope angle from horizontal in **degrees** (positive downhill)
* `downhill_distance` — distance along slope (m), e.g. 8000 for 8 km
* `total_mass`, `static_mass_front/rear`, `axle_distance`, `cog_height` — same as 1g file

In [ ]:
g = 9.81

# ── Downhill — EDIT THESE ──
constant_speed = 25.0      # m/s  (90 km/h example)
downhill_angle = 5.0       # degrees from horizontal
downhill_distance = 8000.0 # m along slope (8 km)

# ── Vehicle — EDIT THESE ──
total_mass = 272.0              # kg
static_mass_front = 177.64      # kg
static_mass_rear = 94.36        # kg
axle_distance = 2.3             # m wheelbase L
cog_height = 19.90 * 0.0254     # m height h normal to slope

use_widgets = False

print(f"speed={constant_speed} m/s  angle={downhill_angle}°  distance={downhill_distance/1000:.1f} km")
print(f"mass={total_mass} kg  front={static_mass_front} kg  rear={static_mass_rear} kg  L={axle_distance} m  h={cog_height:.4f} m")

In [ ]:
if use_widgets and HAS_WIDGETS:
    w_angle = widgets.FloatSlider(value=downhill_angle, min=1, max=15, step=0.5, description='angle°')
    w_speed = widgets.FloatSlider(value=constant_speed, min=5, max=35, step=1, description='speed')
    w_dist  = widgets.FloatSlider(value=downhill_distance, min=1000, max=20000, step=500, description='dist m')
    display(w_angle, w_speed, w_dist)
    print("Adjust sliders, then reassign: downhill_angle=w_angle.value etc. and re-run below.")
else:
    print("Widgets off — using direct values." if not use_widgets else "ipywidgets missing — pip install ipywidgets")

## 2 — Secondary Variables & Slope Decomposition

Weight is split into normal and parallel components:

$$
W_{\perp}=W\cos\theta,\quad W_{\parallel}=W\sin\theta,\quad B_{total}=W_{\parallel}
$$

In [ ]:
assert total_mass is not None and static_mass_front is not None and downhill_angle is not None
assert downhill_distance is not None and constant_speed is not None
assert abs((static_mass_front+static_mass_rear)-total_mass) < 1.0, "Masses must sum to total"

total_weight = total_mass * g
static_weight_front = static_mass_front * g
static_weight_rear  = static_mass_rear * g

front_to_cog = axle_distance * static_mass_rear / total_mass
rear_to_cog  = axle_distance - front_to_cog  # == L * static_mass_front / total

downhill_angle_rad = math.radians(downhill_angle)
weight_normal_total   = total_weight * math.cos(downhill_angle_rad)
weight_parallel_total = total_weight * math.sin(downhill_angle_rad)

braking_force_total = weight_parallel_total  # steady state
braking_power_total = braking_force_total * constant_speed

print(f"total_weight = {total_weight:,.1f} N")
print(f"W_normal = {weight_normal_total:,.1f} N   W_parallel = {weight_parallel_total:,.1f} N  (theta={downhill_angle}°)")
print(f"front_to_cog={front_to_cog:.3f} m  rear_to_cog={rear_to_cog:.3f} m")
print(f"braking_force_total = {braking_force_total:,.1f} N  (balances gravity)")
print(f"braking_power_total = {braking_power_total:,.0f} W  ({braking_power_total/1000:.2f} kW)")

## 3 — Dynamic Weight Transfer on Grade

Moment about rear contact patch (steady, $a=0$):

$$
N_{1}L = W\cos\theta \cdot b + W\sin\theta \cdot h
$$
$$N_{1}=\frac{W\cos\theta\,b+W\sin\theta\,h}{L},\quad N_{2}=W\cos\theta-N_{1}$$

Ideal braking distribution is proportional to normal loads:
$$B_{1}=B_{tot}\frac{N_{1}}{W\cos\theta},\quad P_{1}=P_{tot}\frac{N_{1}}{W\cos\theta}$$

In [ ]:
dynamic_weight_front = (total_weight*math.cos(downhill_angle_rad)*rear_to_cog + total_weight*math.sin(downhill_angle_rad)*cog_height) / axle_distance
dynamic_weight_rear  = weight_normal_total - dynamic_weight_front

dynamic_braking_force_front = braking_force_total * dynamic_weight_front / weight_normal_total
dynamic_braking_force_rear  = braking_force_total - dynamic_braking_force_front

power_front = braking_power_total * dynamic_weight_front / weight_normal_total
power_rear  = braking_power_total - power_front

heat_rate_front = power_front
heat_rate_rear  = power_rear

descent_time = downhill_distance / constant_speed
energy_total = braking_force_total * downhill_distance  # also power*time
energy_front = energy_total * dynamic_weight_front / weight_normal_total
energy_rear  = energy_total - energy_front

heat_front = energy_front
heat_rear  = energy_rear
heat_total = energy_total

print("── Normal loads on slope ──")
print(f"dynamic_weight_front = {dynamic_weight_front:,.1f} N  ({dynamic_weight_front/weight_normal_total*100:.1f}% of W_normal)")
print(f"dynamic_weight_rear  = {dynamic_weight_rear:,.1f} N  ({dynamic_weight_rear/weight_normal_total*100:.1f}%)")
print(f"sum = {dynamic_weight_front+dynamic_weight_rear:,.1f} N vs W_normal {weight_normal_total:,.1f} N")
print()
print(f"braking_force_front = {dynamic_braking_force_front:,.1f} N  ({dynamic_braking_force_front/braking_force_total*100:.1f}%)")
print(f"braking_force_rear  = {dynamic_braking_force_rear:,.1f} N  ({dynamic_braking_force_rear/braking_force_total*100:.1f}%)")
print()
print(f"power_front = {power_front:,.0f} W  ({power_front/braking_power_total*100:.1f}%)")
print(f"power_rear  = {power_rear:,.0f} W  ({power_rear/braking_power_total*100:.1f}%)")
print(f"power_total = {braking_power_total:,.0f} W")
print()
print(f"descent_time = {descent_time:,.0f} s  ({descent_time/60:.1f} min)")
print(f"energy_total = {energy_total/1e6:.2f} MJ  ({energy_total/3.6e6:.2f} kWh)")
print(f"energy_front = {energy_front/1e6:.2f} MJ  ({energy_front/energy_total*100:.1f}%)  — {energy_front/2/1e6:.2f} MJ per front rotor")
print(f"energy_rear  = {energy_rear/1e6:.2f} MJ  ({energy_rear/energy_total*100:.1f}%)")
print(f"avg heat rate front = {heat_rate_front:,.0f} W  rear = {heat_rate_rear:,.0f} W")

## 4 — Visualisation

In [ ]:
labels = ['Front axle', 'Rear axle']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

n_vals = [dynamic_weight_front, dynamic_weight_rear]
axes[0].bar(labels, n_vals, color=['#1f77b4','#ff7f0e'])
axes[0].set_title(f'Normal Load on {downhill_angle}° Slope (N)')
for i, v in enumerate(n_vals):
    axes[0].text(i, v, f"{v:,.0f}\n({v/weight_normal_total*100:.1f}%)", ha='center', va='bottom', fontsize=9)

p_vals = [power_front, power_rear]
axes[1].bar(labels, p_vals, color=['#1f77b4','#ff7f0e'])
axes[1].set_title('Braking Power Distribution (W)')
for i, v in enumerate(p_vals):
    axes[1].text(i, v, f"{v:,.0f} W\n({v/braking_power_total*100:.1f}%)", ha='center', va='bottom', fontsize=9)

e_vals = [energy_front/1e6, energy_rear/1e6]
axes[2].bar(labels, e_vals, color=['#1f77b4','#ff7f0e'])
axes[2].set_title(f'Total Energy over {downhill_distance/1000:.0f} km (MJ)')
for i, v in enumerate(e_vals):
    axes[2].text(i, v, f"{v:.2f} MJ\n({v/sum(e_vals)*100:.1f}%)", ha='center', va='bottom', fontsize=9)

fig.suptitle(f'Continuous Downhill — {constant_speed} m/s, {downhill_angle}° slope, L={axle_distance}m h={cog_height:.3f}m', fontsize=11)
plt.tight_layout()
plt.show()

# per-rotor power (2 rotors per axle) — useful for thermal sizing
print(f"Per front rotor: {power_front/2:,.0f} W  |  Per rear rotor: {power_rear/2:,.0f} W")
print(f"Per front rotor energy: {energy_front/2/1e6:.2f} MJ  |  Per rear rotor: {energy_rear/2/1e6:.2f} MJ")

## 5 — Sensitivity Sweeps

In [ ]:
# sweep angle
angles = np.linspace(1, 12, 50)
p_tot = []
front_pct = []
for ang in angles:
    rad = math.radians(ang)
    Wn = total_weight * math.cos(rad)
    Wp = total_weight * math.sin(rad)
    N1 = (total_weight*math.cos(rad)*rear_to_cog + total_weight*math.sin(rad)*cog_height)/axle_distance
    p_tot.append(Wp * constant_speed / 1000)  # kW
    front_pct.append(N1 / Wn * 100)

fig, ax1 = plt.subplots(figsize=(8,4))
ax1.set_xlabel('Downhill angle (°)')
ax1.set_ylabel('Total braking power (kW)', color='tab:blue')
ax1.plot(angles, p_tot, color='tab:blue', label='Power (kW)')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.axvline(downhill_angle, color='gray', linestyle='--', label=f'current {downhill_angle}°')
ax2 = ax1.twinx()
ax2.set_ylabel('Front normal %', color='tab:orange')
ax2.plot(angles, front_pct, color='tab:orange', linestyle='--', label='Front %')
ax2.tick_params(axis='y', labelcolor='tab:orange')
plt.title(f'Power & front bias vs slope @ {constant_speed} m/s')
ax1.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# sweep speed
speeds = np.linspace(5, 35, 50)
pwr_speed = [(total_weight*math.sin(downhill_angle_rad))*v/1000 for v in speeds]
plt.figure(figsize=(7,4))
plt.plot(speeds, pwr_speed)
plt.axvline(constant_speed, color='red', linestyle='--', label=f'{constant_speed} m/s')
plt.xlabel('Constant speed (m/s)')
plt.ylabel('Total braking power (kW)')
plt.title(f'Power vs speed @ {downhill_angle}° slope')
plt.legend(); plt.grid(alpha=0.3); plt.show()

---
*Notebook generated from `continuous downhill.py`.*